# Skin Condition Classifier — Full Pipeline
## Data Preprocessing + Model Training

This notebook runs the complete pipeline from raw datasets to a trained model ready for deployment.

**Before running:** Make sure T4 GPU is selected — Runtime → Change runtime type → T4 GPU

**Then click Runtime → Run all and walk away. The whole thing takes about 45-60 minutes.**

The only interruption is Cell 1 which asks you to upload kaggle.json.

---
# Part 1 — Data Preprocessing

## Cell 1 — Upload Kaggle API key

In [ ]:
from google.colab import files
import os

print('Please upload your kaggle.json file...')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API key configured.')

## Cell 2 — Install packages and download datasets

In [ ]:
!pip install kaggle --quiet

os.makedirs('/content/data/acne04', exist_ok=True)
os.makedirs('/content/data/dermnet', exist_ok=True)

print('Downloading ACNE04...')
!kaggle datasets download -d jincyjis/acne04 -p /content/data/acne04 --unzip
print('ACNE04 done.')

print('Downloading DermNet...')
!kaggle datasets download -d shubhamgoel27/dermnet -p /content/data/dermnet --unzip
print('DermNet done.')

## Cell 3 — Import all libraries

In [ ]:
import os
import time
import copy
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('All libraries loaded.')

## Cell 4 — Process ACNE04 dataset
Images are labeled by filename prefix (levle0 = clear, levle1 = mild, levle2 = moderate, levle3 = severe).

In [ ]:
PROCESSED_DIR = Path('/content/data/processed')
IMG_SIZE = (224, 224)

acne_root = Path('/content/data/acne04/JPEGImages')
all_acne_imgs = list(acne_root.glob('*.jpg'))

severity_map = {
    'levle0': 'acne_clear',
    'levle1': 'acne_mild',
    'levle2': 'acne_moderate',
    'levle3': 'acne_severe'
}

print('Processing ACNE04...')
acne_summary = {}
random.seed(42)

for prefix, label in severity_map.items():
    imgs = [p for p in all_acne_imgs if p.name.startswith(prefix)]
    dest = PROCESSED_DIR / label
    dest.mkdir(parents=True, exist_ok=True)
    selected = random.sample(imgs, min(400, len(imgs)))
    saved = 0
    for i, path in enumerate(selected):
        try:
            img = Image.open(path).convert('RGB')
            img = img.resize(IMG_SIZE, Image.LANCZOS)
            img.save(dest / f'{label}_{i:04d}.jpg', 'JPEG', quality=90)
            saved += 1
        except:
            pass
    acne_summary[label] = saved
    print(f'  {label}: {saved} images')

print('ACNE04 done.')

## Cell 5 — Process DermNet dataset
Selects 11 relevant everyday skin condition classes and caps each at 1000 images.

In [ ]:
dermnet_train = Path('/content/data/dermnet/train')
dermnet_counts = {}
for folder in sorted(dermnet_train.iterdir()):
    if folder.is_dir():
        imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
        dermnet_counts[folder.name] = len(imgs)

SELECTED_CLASSES = [
    'Acne and Rosacea Photos',
    'Atopic Dermatitis Photos',
    'Eczema Photos',
    'Psoriasis pictures Lichen Planus and related diseases',
    'Seborrheic Keratoses and other Benign Tumors',
    'Nail Fungus and other Nail Disease',
    'Urticaria Hives',
    'Warts Molluscum and other Viral Infections',
    'Tinea Ringworm Candidiasis and other Fungal Infections',
    'Cellulitis Impetigo and other Bacterial Infections',
    'Poison Ivy Photos and other Contact Dermatitis',
]

CLASS_LABEL_MAP = {
    'Acne and Rosacea Photos': 'acne_rosacea',
    'Atopic Dermatitis Photos': 'atopic_dermatitis',
    'Eczema Photos': 'eczema',
    'Psoriasis pictures Lichen Planus and related diseases': 'psoriasis',
    'Seborrheic Keratoses and other Benign Tumors': 'seborrheic_keratoses',
    'Nail Fungus and other Nail Disease': 'nail_fungus',
    'Urticaria Hives': 'urticaria',
    'Warts Molluscum and other Viral Infections': 'warts',
    'Tinea Ringworm Candidiasis and other Fungal Infections': 'tinea_ringworm',
    'Cellulitis Impetigo and other Bacterial Infections': 'cellulitis',
    'Poison Ivy Photos and other Contact Dermatitis': 'contact_dermatitis',
}

def process_and_save(src_paths, dest_folder, max_images=1000):
    dest_folder.mkdir(parents=True, exist_ok=True)
    paths = random.sample(src_paths, min(max_images, len(src_paths)))
    saved = 0
    for i, path in enumerate(paths):
        try:
            img = Image.open(path).convert('RGB')
            img = img.resize(IMG_SIZE, Image.LANCZOS)
            img.save(dest_folder / f'{dest_folder.name}_{i:04d}.jpg', 'JPEG', quality=90)
            saved += 1
        except:
            pass
    return saved

print('Processing DermNet classes...')
dermnet_summary = {}

for cls in SELECTED_CLASSES:
    label = CLASS_LABEL_MAP[cls]
    all_imgs = []
    for split in ['train', 'test']:
        folder = Path(f'/content/data/dermnet/{split}') / cls
        if folder.exists():
            all_imgs += list(folder.glob('*.jpg')) + list(folder.glob('*.png')) + list(folder.glob('*.jpeg'))
    if all_imgs:
        n = process_and_save(all_imgs, PROCESSED_DIR / label)
        dermnet_summary[label] = n
        print(f'  {label}: {n} images')

total = sum(acne_summary.values()) + sum(dermnet_summary.values())
print(f'\nTotal processed: {total} images across {len(acne_summary) + len(dermnet_summary)} classes')

## Cell 6 — Train / Val / Test split

In [ ]:
SPLIT_DIR = Path('/content/data/split')
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

random.seed(42)
split_summary = {'train': {}, 'val': {}, 'test': {}}

for class_folder in sorted(PROCESSED_DIR.iterdir()):
    if not class_folder.is_dir():
        continue
    class_name = class_folder.name
    all_imgs = list(class_folder.glob('*.jpg'))
    random.shuffle(all_imgs)
    n = len(all_imgs)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    splits = {
        'train': all_imgs[:n_train],
        'val':   all_imgs[n_train:n_train + n_val],
        'test':  all_imgs[n_train + n_val:]
    }
    for split_name, split_imgs in splits.items():
        dest = SPLIT_DIR / split_name / class_name
        dest.mkdir(parents=True, exist_ok=True)
        for img_path in split_imgs:
            shutil.copy(img_path, dest / img_path.name)
        split_summary[split_name][class_name] = len(split_imgs)

print('Split complete:')
for split_name, classes in split_summary.items():
    print(f'  {split_name}: {sum(classes.values())} images across {len(classes)} classes')

## Cell 7 — Save class names

In [ ]:
class_names_list = sorted([f.name for f in (SPLIT_DIR / 'train').iterdir() if f.is_dir()])

with open('/content/data/class_names.txt', 'w') as f:
    for name in class_names_list:
        f.write(name + '\n')

print('Class names:')
for i, name in enumerate(class_names_list):
    print(f'  {i}: {name}')

---
# Part 2 — Model Training

## Cell 8 — Define data transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}
print('Transforms defined.')

## Cell 9 — Load datasets and create DataLoaders

In [ ]:
BATCH_SIZE = 32

image_datasets = {
    split: datasets.ImageFolder(
        root=str(SPLIT_DIR / split),
        transform=data_transforms[split]
    )
    for split in ['train', 'val', 'test']
}

dataloaders = {
    split: DataLoader(
        image_datasets[split],
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        num_workers=2,
        pin_memory=True
    )
    for split in ['train', 'val', 'test']
}

class_names = image_datasets['train'].classes
NUM_CLASSES = len(class_names)

print(f'Classes ({NUM_CLASSES}): {class_names}')
print(f'Train batches: {len(dataloaders["train"])}')
print(f'Val batches:   {len(dataloaders["val"])}')
print(f'Test batches:  {len(dataloaders["test"])}')

## Cell 10 — Load MobileNetV2 and set up classifier head

In [ ]:
model = models.mobilenet_v2(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 512),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(512, NUM_CLASSES)
)

model = model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total:,}')
print(f'Trainable parameters: {trainable:,} ({100*trainable/total:.1f}%)')
print('Base model frozen. Only classifier head will train in Phase A.')

## Cell 11 — Loss function, optimizer, scheduler

In [ ]:
class_counts = []
for cls in class_names:
    cls_path = SPLIT_DIR / 'train' / cls
    class_counts.append(len(list(cls_path.glob('*.jpg'))))

class_counts  = torch.tensor(class_counts, dtype=torch.float)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

print('Loss, optimizer and scheduler ready.')

## Cell 12 — Training function

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs, phase_name):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}', end=' | ')
        start = time.time()

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc  = running_corrects.double() / len(image_datasets[phase])
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'train':
                print(f'Train loss: {epoch_loss:.4f} acc: {epoch_acc:.4f}', end=' | ')
            else:
                elapsed = time.time() - start
                print(f'Val loss: {epoch_loss:.4f} acc: {epoch_acc:.4f} ({elapsed:.0f}s)')
                scheduler.step(epoch_loss)
                if epoch_acc > best_val_acc:
                    best_val_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())

    print(f'\nBest val accuracy ({phase_name}): {best_val_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model, history

print('Training function defined.')

## Cell 13 — Phase A: Train classifier head (10 epochs)
Base model is frozen. Only the new classification head trains here. Expect ~15-20 minutes.

In [ ]:
print('Phase A: Training classifier head only...')
print('-' * 70)
model, history_a = train_model(model, criterion, optimizer, scheduler,
                                num_epochs=10, phase_name='Phase A')

## Cell 14 — Phase B: Fine-tune top layers (10 epochs)
Unfreezes the top layers of MobileNetV2 and trains with a lower learning rate. Expect ~20-25 minutes.

In [ ]:
for i, layer in enumerate(model.features):
    if i >= 15:
        for param in layer.parameters():
            param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters after unfreezing: {trainable:,}')

optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)
scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='min', factor=0.5, patience=2, verbose=True
)

print('Phase B: Fine-tuning top layers...')
print('-' * 70)
model, history_b = train_model(model, criterion, optimizer_ft, scheduler_ft,
                                num_epochs=10, phase_name='Phase B')

## Cell 15 — Plot training curves

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history['train_loss'], label='Train', marker='o')
    ax1.plot(history['val_loss'],   label='Val',   marker='o')
    ax1.set_title('Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.plot(history['train_acc'], label='Train', marker='o')
    ax2.plot(history['val_acc'],   label='Val',   marker='o')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_history(history_a, 'Phase A — Classifier head')
plot_history(history_b, 'Phase B — Fine-tuning')

## Cell 16 — Evaluate on test set

In [ ]:
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc   = np.mean(all_preds == all_labels)

print(f'Test accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')
print()
print(classification_report(all_labels, all_preds, target_names=class_names))

## Cell 17 — Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(14, 12))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion matrix (normalized)')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## Cell 18 — Save and download the trained model
Save this file to your local `model/` folder when it downloads.

In [ ]:
os.makedirs('/content/model', exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'num_classes': NUM_CLASSES,
    'architecture': 'mobilenet_v2',
    'test_accuracy': float(test_acc),
}, '/content/model/skin_classifier.pt')

size_mb = os.path.getsize('/content/model/skin_classifier.pt') / 1e6
print(f'Model saved: skin_classifier.pt ({size_mb:.1f} MB)')
print(f'Test accuracy: {test_acc*100:.1f}%')

from google.colab import files
files.download('/content/model/skin_classifier.pt')
print('Downloading model file — save it to your local model/ folder.')

---
## Pipeline complete

Your trained model is at `model/skin_classifier.pt` on your local machine.

**Healthy results to look for:**
- Overall test accuracy above 60% is good, above 70% is strong for 15 classes
- Training curves should show loss going down and accuracy going up
- Confusion matrix diagonal should be mostly bright

**Next step:** Phase 4 — build the Streamlit app.